# Notebook 3 — Interactive Visualization

Applies a trained model from notebook 2 to any preprocessed city and shows
the result on an interactive map over RGB satellite imagery. **No Earth
Engine and no GPU needed** — the model, its normalization statistics and its
threshold all travel inside the checkpoint, and the imagery patches come
from Drive.

Map layers (toggleable in the layer control):

* **ground truth** — building footprints coloured by the UNOSAT label
* **prediction confidence** — footprints coloured by model probability
* **disagreements** — false positives and false negatives outlined in cyan
* Buildings that were in the model's **training or validation region** are
  marked in the tooltip (`split` field) — judge the model only on `test`
  or `unused` buildings, its training area is trivially easy for it.

## 1. Setup — pick an experiment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
BASE = "/content/drive/MyDrive/War-Damage-Detection"   # adjust if your folder differs
sys.path.append(BASE)

import os
import numpy as np
import geopandas as gpd
import torch

from pipeline import (CITY_REGISTRY, EXPERIMENTS_DIR,
                      load_processed, select_channels, split_assignment,
                      spatial_smooth, build_model, predict_probs,
                      experiment_dirs)

# ---- choose which experiment to visualize -------------------------------
EXPERIMENT_NAME = "exp001_small_cnn_sar"
# -------------------------------------------------------------------------

print("experiments on Drive:", sorted(os.listdir(EXPERIMENTS_DIR)))

EXP_ROOT, OUT = experiment_dirs(EXPERIMENT_NAME)
ckpt_files = [f for f in os.listdir(OUT["models"]) if f.endswith(".pt")]
if not ckpt_files:
    raise FileNotFoundError(f"no checkpoint in {OUT['models']}, "
                            f"run notebook 2 first")

ckpt = torch.load(os.path.join(OUT["models"], ckpt_files[0]),
                  map_location="cpu", weights_only=False)
cfg = ckpt["config"]

model = build_model(cfg["model"], ckpt["channel_names"])
model.load_state_dict(ckpt["state_dict"])
model.eval()

print(f"loaded {ckpt_files[0]}: best epoch {ckpt['best_epoch']}, "
      f"val PR AUC {ckpt['val_ap']:.4f}, threshold {ckpt['val_threshold']:.3f}")

## 2. Predict a city

Scores every preprocessed building of a city and writes a GeoJSON to the
experiment's `predictions/` folder — ready for QGIS or any other GIS tool.
Each building carries its probability, its predicted and true label, the
PWTT statistic, and which split region it belonged to during training.

If the experiment used spatial smoothing, it is applied here too, so the map
shows the same scores the evaluation measured. (For a whole-city map the
smoothing runs over the full city at once — fine for visual inspection, but
remember that metrics on smoothed scores near the train/test boundary are
only valid in notebook 2, where smoothing stays inside each region.)

In [ ]:
def predict_city(city, date=None):
    """Score one city with the loaded model and save the result as GeoJSON."""
    d = load_processed(city, date)
    Xn = (select_channels(d, cfg["features"]) - ckpt["mu"]) / ckpt["sd"]
    probs = predict_probs(model, Xn)

    sm = cfg["spatial_smoothing"]
    if sm["enabled"]:
        probs = spatial_smooth(d["xy"], probs, k=sm["k"], weight=sm["weight"])

    out = d["gdf"][["geometry", "class", "max_change"]].copy()
    out["prob"] = probs.round(4)
    out["pred"] = (probs >= ckpt["val_threshold"]).astype(int)
    out["split"] = split_assignment(d["lat"], city, cfg["split"])

    path = os.path.join(OUT["predictions"], f"{city}_{d['date']}.geojson")
    out.to_file(path, driver="GeoJSON")
    print(f"wrote {path}")
    return out


# score one city here, or loop over CITY_REGISTRY to score them all
predictions = predict_city("Gaza")
predictions.head(3)

## 3. The interactive map

Runs entirely on saved GeoJSON files, so once predictions exist this section
works on its own. Set `CITY` (and optionally `DATE`) below and run the cell.
Large cities are subsampled to keep the map responsive.

In [ ]:
import folium
import matplotlib


def prob_to_color(p):
    return matplotlib.colors.to_hex(matplotlib.colormaps["RdYlGn_r"](float(p)))


def build_comparison_map(pred_gdf, max_features=4000):
    """Folium map with ground truth, prediction and disagreement layers."""
    g = pred_gdf
    if len(g) > max_features:
        g = g.sample(max_features, random_state=0)
    g = g.copy()
    g["color_pred"] = [prob_to_color(p) for p in g["prob"]]
    g["color_true"] = np.where(g["class"] == 1, "#C0392B", "#4C72B0")
    agree = g["pred"] == g["class"]

    center = [g.geometry.centroid.y.mean(), g.geometry.centroid.x.mean()]
    m = folium.Map(location=center, zoom_start=14, tiles=None)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
              "World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery", name="RGB satellite").add_to(m)

    fields = ["prob", "class", "pred", "max_change", "split"]
    aliases = ["confidence", "true label", "predicted", "PWTT max_change", "split"]

    def add_layer(frame, color_col, name, show, weight=1, fill=0.7, outline=None):
        folium.GeoJson(
            frame[["geometry"] + fields + [color_col]].to_json(),
            name=name, show=show,
            style_function=lambda f, cc=color_col, w=weight, fo=fill, ol=outline: {
                "color": ol or f["properties"][cc],
                "fillColor": f["properties"][cc],
                "weight": w,
                "fillOpacity": fo,
            },
            tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases),
        ).add_to(m)

    add_layer(g, "color_true", "ground truth", show=False)
    add_layer(g, "color_pred", "prediction confidence", show=True)
    disagreements = g[~agree]
    if len(disagreements):
        add_layer(disagreements, "color_pred", "disagreements", show=True,
                  weight=3, fill=0.0, outline="#00FFFF")
    folium.LayerControl(collapsed=False).add_to(m)
    return m


# ---- choose what to show ------------------------------------------------
CITY = "Gaza"
DATE = None          # None = newest available prediction for this city
# -------------------------------------------------------------------------

files = sorted(f for f in os.listdir(OUT["predictions"])
               if f.startswith(CITY) and f.endswith(".geojson"))
if not files:
    raise FileNotFoundError(f"no predictions for {CITY}, run Section 2 first")
print("available prediction files:", files)

fname = f"{CITY}_{DATE}.geojson" if DATE else files[-1]
print("showing:", fname)
gdf = gpd.read_file(os.path.join(OUT["predictions"], fname))
build_comparison_map(gdf)

## Ideas from here

* Score every registered city (loop `predict_city` over `CITY_REGISTRY`) and
  flip through them by changing `CITY`.
* When a city has predictions for several assessment dates, set `DATE` to
  compare them — with temporal label propagation enabled in notebook 2 this
  becomes a damage *timeline*.
* The GeoJSON files open directly in QGIS or kepler.gl for fancier
  cartography, and they are what a public-facing "living damage map" would
  be built from.